# Exercise 7: Chunk Overlap Experiment

Test how **overlap** between chunks affects retrieval of information that spans chunk boundaries.

**Chunk size fixed at 512. Overlap values:** 0, 64, 128, 256

⚠️ This exercise takes a long time — run only on Colab T4 or better.


In [7]:
# Install required packages
try:
    ip = get_ipython()
    ip.run_line_magic('pip', 'install -q torch transformers sentence-transformers faiss-cpu pymupdf accelerate')
except NameError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch', 'transformers', 'sentence-transformers', 'faiss-cpu', 'pymupdf', 'accelerate'])


In [8]:
import os, sys
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
from typing import List, Tuple
from pathlib import Path

def detect_environment():
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

def get_device():
    if torch.cuda.is_available():
        device, dtype = 'cuda', torch.float16
        print(f"✓ CUDA: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
    elif torch.backends.mps.is_available():
        device, dtype = 'mps', torch.float32
        print("✓ Apple Silicon MPS")
    else:
        device, dtype = 'cpu', torch.float32
        print("⚠ CPU only")
    return device, dtype

ENVIRONMENT = detect_environment()
DEVICE, DTYPE = get_device()
print(f"Environment: {ENVIRONMENT.upper()} | Device: {DEVICE} | Dtype: {DTYPE}")


✓ CUDA: Tesla T4 (15.6 GB)
Environment: COLAB | Device: cuda | Dtype: torch.float16


In [9]:
from pathlib import Path

CORPUS_SUBFOLDER = "ModelTService"  # <- change if needed
DRIVE_BASE = '/content/drive/MyDrive/Colab_Projects/Week05-RAG/Corpora'

if ENVIRONMENT == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    DOC_FOLDER = f"{DRIVE_BASE}/{CORPUS_SUBFOLDER}"
else:
    DOC_FOLDER = str(Path("Corpora") / CORPUS_SUBFOLDER)

print(f"DOC_FOLDER = {DOC_FOLDER}")
assert Path(DOC_FOLDER).exists(), f"Folder not found: {DOC_FOLDER}"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DOC_FOLDER = /content/drive/MyDrive/Colab_Projects/Week05-RAG/Corpora/ModelTService


In [10]:
import fitz

def load_text_file(fp):
    with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def load_pdf_file(fp):
    doc = fitz.open(fp)
    parts = []
    for i, page in enumerate(doc):
        t = page.get_text()
        if t.strip():
            parts.append(f"\n[Page {i+1}]\n{t}")
    doc.close()
    return "\n".join(parts)

def load_documents(folder):
    docs = []
    for fp in Path(folder).rglob("*"):
        if not fp.is_file(): continue
        if fp.suffix.lower() not in ('.pdf', '.txt', '.md'): continue
        try:
            content = load_pdf_file(str(fp)) if fp.suffix.lower() == '.pdf' else load_text_file(str(fp))
            if content.strip():
                docs.append((fp.name, content))
                print(f"✓ {fp.name} ({len(content):,} chars)")
        except Exception as e:
            print(f"✗ {fp.name}: {e}")
    return docs

documents = load_documents(DOC_FOLDER)
print(f"\nLoaded {len(documents)} documents")


✓ Ford-Model-T-Man-1919.txt (95,574 chars)
✓ ModelT-01-10.txt (18,676 chars)
✓ ModelT-11-20.txt (19,009 chars)
✓ ModelT-21-30.txt (17,050 chars)
✓ ModelT-31-40.txt (12,194 chars)
✓ ModelT-41-50.txt (14,264 chars)
✓ ModelT-51-60.txt (14,168 chars)
✓ ModelT-61-62.txt (201 chars)
✓ Ford-Model-T-Man-1919-ocr.pdf (95,517 chars)
✓ ModelT-01-10-ocr.pdf (18,658 chars)
✓ ModelT-11-20-ocr.pdf (19,003 chars)
✓ ModelT-21-30-ocr.pdf (17,025 chars)
✓ ModelT-31-40-ocr.pdf (12,201 chars)
✓ ModelT-41-50-ocr.pdf (14,270 chars)
✓ ModelT-51-60-ocr.pdf (14,107 chars)
✓ ModelT-61-62-ocr.pdf (204 chars)

Loaded 16 documents


In [11]:
from dataclasses import dataclass

@dataclass
class Chunk:
    text: str
    source_file: str
    chunk_index: int
    start_char: int
    end_char: int

def chunk_text(text, source_file, chunk_size=512, chunk_overlap=128):
    chunks, start, idx = [], 0, 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            pb = text.rfind('\n\n', start + chunk_size // 2, end)
            if pb != -1:
                end = pb + 2
            else:
                sb = text.rfind('. ', start + chunk_size // 2, end)
                if sb != -1:
                    end = sb + 2
        s = text[start:end].strip()
        if s:
            chunks.append(Chunk(s, source_file, idx, start, end))
            idx += 1
        prev_start = start
        start = end - chunk_overlap
        if chunks and start <= chunks[-1].start_char:
            start = end
    return chunks

# Default chunking parameters (override per exercise)
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 128

all_chunks = []
for fname, content in documents:
    dc = chunk_text(content, fname, CHUNK_SIZE, CHUNK_OVERLAP)
    all_chunks.extend(dc)
    print(f"  {fname}: {len(dc)} chunks")
print(f"\nTotal chunks: {len(all_chunks)}")


  Ford-Model-T-Man-1919.txt: 326 chunks
  ModelT-01-10.txt: 64 chunks
  ModelT-11-20.txt: 66 chunks
  ModelT-21-30.txt: 56 chunks
  ModelT-31-40.txt: 44 chunks
  ModelT-41-50.txt: 51 chunks
  ModelT-51-60.txt: 46 chunks
  ModelT-61-62.txt: 1 chunks
  Ford-Model-T-Man-1919-ocr.pdf: 316 chunks
  ModelT-01-10-ocr.pdf: 63 chunks
  ModelT-11-20-ocr.pdf: 61 chunks
  ModelT-21-30-ocr.pdf: 56 chunks
  ModelT-31-40-ocr.pdf: 44 chunks
  ModelT-41-50-ocr.pdf: 47 chunks
  ModelT-51-60-ocr.pdf: 44 chunks
  ModelT-61-62-ocr.pdf: 1 chunks

Total chunks: 1286


In [12]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading: {EMBEDDING_MODEL} on {DEVICE}")
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
EMBEDDING_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dim: {EMBEDDING_DIM}")


Loading: sentence-transformers/all-MiniLM-L6-v2 on cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dim: 384


In [13]:
import faiss, pickle
from pathlib import Path

# Cache config — CACHE_KEY encodes corpus + chunk params
# Change it if you change CORPUS_SUBFOLDER, CHUNK_SIZE, or CHUNK_OVERLAP
CACHE_DIR   = Path("/content/drive/MyDrive/Colab_Projects/Week05-RAG/cache")
CACHE_KEY   = "modelTService_512_128"
CHUNKS_FILE = CACHE_DIR / f"{CACHE_KEY}.chunks.pkl"
EMBEDS_FILE = CACHE_DIR / f"{CACHE_KEY}.embeddings.npy"
INDEX_FILE  = CACHE_DIR / f"{CACHE_KEY}.faiss"
CACHE_DIR.mkdir(exist_ok=True)

def save_cache():
    with open(CHUNKS_FILE, "wb") as f: pickle.dump(all_chunks, f)
    np.save(str(EMBEDS_FILE), chunk_embeddings)
    faiss.write_index(index, str(INDEX_FILE))
    print(f"✓ Cache saved → {CACHE_DIR}/{CACHE_KEY}.*")

def load_cache():
    global all_chunks, chunk_embeddings, index
    with open(CHUNKS_FILE, "rb") as f: all_chunks = pickle.load(f)
    chunk_embeddings = np.load(str(EMBEDS_FILE))
    index = faiss.read_index(str(INDEX_FILE))
    print(f"✓ Cache loaded: {len(all_chunks)} chunks, {index.ntotal} vectors")

if CHUNKS_FILE.exists() and EMBEDS_FILE.exists() and INDEX_FILE.exists():
    load_cache()
else:
    print("No cache found — embedding chunks (first run only, will be cached after)...")
    chunk_embeddings = embed_model.encode(
        [c.text for c in all_chunks], show_progress_bar=True
    ).astype("float32")
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    faiss.normalize_L2(chunk_embeddings)
    index.add(chunk_embeddings)
    print(f"✓ Index built: {index.ntotal} vectors")
    save_cache()

def rebuild_pipeline(chunk_size=512, chunk_overlap=128):
    """Re-chunk, re-embed, rebuild index in-memory (for Ex 7 & 8 experiments)."""
    global all_chunks, chunk_embeddings, index
    all_chunks = []
    for fname, content in documents:
        all_chunks.extend(chunk_text(content, fname, chunk_size, chunk_overlap))
    chunk_embeddings = embed_model.encode(
        [c.text for c in all_chunks], show_progress_bar=True
    ).astype("float32")
    faiss.normalize_L2(chunk_embeddings)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(chunk_embeddings)
    print(f"Rebuilt: {len(all_chunks)} chunks | size={chunk_size}, overlap={chunk_overlap}")

def retrieve(query, top_k=5):
    qe = embed_model.encode([query]).astype("float32")
    faiss.normalize_L2(qe)
    scores, idxs = index.search(qe, top_k)
    return [(all_chunks[i], float(s)) for s, i in zip(scores[0], idxs[0]) if i != -1]


✓ Cache loaded: 1286 chunks, 1286 vectors


In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading LLM: {LLM_MODEL} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

if DEVICE == 'cuda':
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, device_map="auto",
                torch_dtype=DTYPE, trust_remote_code=True)
elif DEVICE == 'mps':
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE,
                trust_remote_code=True).to(DEVICE)
else:
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE,
                trust_remote_code=True)
print("LLM loaded ✓")


Loading LLM: Qwen/Qwen2.5-1.5B-Instruct on cuda...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded ✓


In [15]:
PROMPT_TEMPLATE = """You are a helpful assistant that answers questions based on the provided context.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer ONLY based on the information in the context above.
- If the context doesn't contain the answer, say so explicitly.
- Quote relevant passages to support your answer.
- Be concise and direct.

ANSWER:"""

def generate_response(prompt, max_new_tokens=512, temperature=0.3):
    inputs = tokenizer(prompt, return_tensors="pt")
    if DEVICE == 'cuda':
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    else:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=temperature,
                             do_sample=(temperature > 0),
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def direct_query(question, max_new_tokens=512):
    prompt = f"Answer this question:\n{question}\n\nAnswer:"
    return generate_response(prompt, max_new_tokens)

def rag_query(question, top_k=5, show_context=False, prompt_template=None):
    results = retrieve(question, top_k)
    ctx_parts = [f"[Source: {c.source_file}, Score: {s:.3f}]\n{c.text}" for c, s in results]
    context = "\n\n---\n\n".join(ctx_parts)
    if show_context:
        print("=== RETRIEVED CONTEXT ==="); print(context); print("=" * 40)
    tmpl = prompt_template if prompt_template else PROMPT_TEMPLATE
    return generate_response(tmpl.format(context=context, question=question))


In [16]:
import time

# A question whose answer likely spans a chunk boundary
BOUNDARY_QUESTIONS = [
    "How do I adjust the carburetor on a Model T?",
    "What is the complete procedure for adjusting the transmission bands?",
    "What maintenance does the engine require after long storage?",
]

OVERLAP_VALUES = [0, 64, 128, 256]
CHUNK_SIZE_FIXED = 512

results_overlap = {}

for overlap in OVERLAP_VALUES:
    print(f"\n{'='*70}")
    print(f"OVERLAP = {overlap} (chunk_size={CHUNK_SIZE_FIXED})")
    t0 = time.time()
    rebuild_pipeline(chunk_size=CHUNK_SIZE_FIXED, chunk_overlap=overlap)
    build_time = time.time() - t0
    n_chunks = len(all_chunks)
    print(f"  Build time: {build_time:.1f}s | Chunks: {n_chunks}")

    results_overlap[overlap] = {"build_time": build_time, "n_chunks": n_chunks, "answers": {}}

    for q in BOUNDARY_QUESTIONS:
        t0 = time.time()
        answer = rag_query(q, top_k=5)
        lat = time.time() - t0
        results_overlap[overlap]["answers"][q] = {"answer": answer, "latency": lat}
        print(f"\n  Q: {q}")
        print(f"  A ({lat:.1f}s): {answer[:300]}")

# Summary table
print("\n" + "="*70)
print("SUMMARY: Chunks created per overlap value")
print("="*70)
for ov, data in results_overlap.items():
    print(f"  overlap={ov:3d}: {data['n_chunks']:6d} chunks | build_time={data['build_time']:.1f}s")



OVERLAP = 0 (chunk_size=512)


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Rebuilt: 888 chunks | size=512, overlap=0
  Build time: 2.0s | Chunks: 888

  Q: How do I adjust the carburetor on a Model T?
  A (8.7s): To adjust the carburetor on a Model T, you should:

1. Start the motor.
2. Advance the throttle lever to the sixth notch.
3. Retard the spark about four notches.
4. Cut off the gasoline flow by turning the needle valve to the right until the engine starts to misfire.
5. Gradually increase the gasoli

  Q: What is the complete procedure for adjusting the transmission bands?
  A (15.4s): The complete procedure for adjusting the transmission bands involves several steps:

1. **Positioning**: Place the drum-on-table with the hub in a vertical position. Ensure the platway is placed correctly such that the hub aligns with the gear at the top.

2. **Adjustment**: Next, ensure that the re

  Q: What maintenance does the engine require after long storage?
  A (6.2s): After long storage, the engine requires periodic inspection for any signs of corrosion or rus

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Rebuilt: 1051 chunks | size=512, overlap=64
  Build time: 1.5s | Chunks: 1051

  Q: How do I adjust the carburetor on a Model T?
  A (9.7s): To adjust the carburetor on a Model T, follow these steps:

1. Start the motor.
2. Advance the throttle lever to the sixth notch.
3. Retard the spark about four notches.
4. Cut off the flow of gasoline by rotating the needle valve to the right until the engine begins to misfire.
5. Gradually increas

  Q: What is the complete procedure for adjusting the transmission bands?
  A (23.5s): The complete procedure for adjusting the transmission bands involves:

1. **Checking Bands Adjusted**: Confirm that the bands have been correctly adjusted as per instructions.
2. **Loosening Lock Nut**: Loosen the lock nut at the tight side of the transmission cover.
3. **Adjusting Screw**: Turn the

  Q: What maintenance does the engine require after long storage?
  A (7.3s): After long storage, the engine requires draining the water from the radiator, adding about

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

Rebuilt: 1286 chunks | size=512, overlap=128
  Build time: 2.0s | Chunks: 1286

  Q: How do I adjust the carburetor on a Model T?
  A (5.4s): To adjust the carburetor on a Model T, you should refer to the instructions or diagrams provided in the context. Specifically, look for sections discussing how the needle valve controls the amount of fuel entering the mixture and how the throttle controls the flow of air. Additionally, consider the 

  Q: What is the complete procedure for adjusting the transmission bands?
  A (8.0s): The complete procedure for adjusting the transmission bands involves:

1. **Loosening the Lock Nut**: First, loosen the lock nut at the tight side of the transmission cover.
2. **Adjusting the Screw**: Next, turn the adjusting screw as described in Cut No. 12 to tighten or loosen the speed band.
3. 

  Q: What maintenance does the engine require after long storage?
  A (5.7s): After long storage, the engine requires regular inspection and draining of any accumulated

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Rebuilt: 2016 chunks | size=512, overlap=256
  Build time: 3.3s | Chunks: 2016

  Q: How do I adjust the carburetor on a Model T?
  A (10.5s): To adjust the carburetor on a Model T, you should:

1. Start the motor.
2. Advance the throttle lever to the sixth notch.
3. Retard the spark about four notches.
4. Cut off the gasoline flow by rotating the needle valve to the right until the engine starts to misfire.
5. Gradually increase the gasol

  Q: What is the complete procedure for adjusting the transmission bands?
  A (14.8s): To adjust the transmission bands, follow these steps:

1. **Check Bands Adjusted**: Confirm that the bands have been correctly adjusted as per instructions.
   
2. **Loosen Lock Nut**: Loosen the lock nut at the tight side of the transmission cover.

3. **Adjusting Screw**: Turn the adjusting screw 

  Q: What maintenance does the engine require after long storage?
  A (6.5s): After long storage, the engine requires thorough cleaning and inspection to remove any a

## Analysis

| Overlap | # Chunks | Index Size (relative) | Retrieval Quality (boundary Q) | Redundancy |
|---------|----------|-----------------------|-------------------------------|------------|
| 0 | 888 | 1.0× | ✅ Good carburetor steps; ⚠️ partial transmission; ⚠️ generic storage | Low |
| 64 | 1051 | 1.18× | ✅ Good carburetor; ✅ complete transmission (lock nut + cover); ✅ **best storage** (drain, kerosene, refill) | Low-Medium |
| 128 | 1286 | 1.45× | ⚠️ **Worse carburetor** (vague "refer to instructions"); ✅ transmission; ⚠️ generic storage | Medium |
| 256 | 2016 | 2.27× | ✅ Good carburetor (back to specific steps); ✅ transmission; ⚠️ generic storage | High |

**Does higher overlap improve retrieval of information spanning chunk boundaries?**

Yes, but non-monotonically. overlap=64 was the **sweet spot** — it captured boundary-spanning content for the storage question (draining radiator, kerosene crankcase flush, refilling) that overlap=0 missed. Surprisingly, overlap=128 *degraded* carburetor answer quality — the overlapping duplicate chunks diluted the prompt with redundant partial context, causing the model to respond vaguely instead of following specific instructions.

**Cost of higher overlap:**

- Index size grows from 888 → 2016 chunks (2.27×) going from overlap=0 to overlap=256.
- Build time increases from 2.0s → 3.3s.
- Prompt context becomes increasingly redundant: multiple near-identical overlapping chunks occupy context window space that could hold more diverse evidence.

**Diminishing returns beyond overlap=64:**

overlap=64 → 128 → 256 showed no consistent quality improvement and introduced regression (carburetor at overlap=128). The jump from overlap=0 → 64 was the most impactful, recovering the long-storage procedure that was split across a chunk boundary. **Recommendation: overlap=64–128 for this corpus size and chunk_size=512.**

In [17]:
import pickle

def save_index(path):
    faiss.write_index(index, f"{path}.faiss")
    with open(f"{path}.chunks", 'wb') as f:
        pickle.dump(all_chunks, f)
    print(f"✓ Saved {path}.faiss + .chunks")

save_index("rag_index")


✓ Saved rag_index.faiss + .chunks
